In [1]:
import numpy as np  
import pandas as pd 

In [3]:
df = pd.read_csv(r"F:\Z_SQL&PowerBi Projects\Inventory\Stock_data.csv")
df.head()

,SKU-ID,Description,2020_units_sold,2021_start_stock
0,82486,3 DRAWER ANTIQUE WHITE WOOD CABINET,440,917
1,23435,3 RAFFIA RIBBONS VINTAGE CHRISTMAS,692,1033
2,85034B,3 WHITE CHOC MORRIS BOXED CANDLES,1610,1142
3,84559A,3D SHEET OF DOG STICKERS,918,620
4,23697,A PRETTY THANK YOU CARD,557,530


In [53]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   item_id           104 non-null    object
 1   Description       104 non-null    object
 2   2020_units_sold   104 non-null    int64 
 3   2021_start_stock  104 non-null    int64 
dtypes: int64(2), object(2)
memory usage: 3.4+ KB


### Uderstanding Structure and Cleaning data

In [19]:
df = df.rename(columns = {'SKU-ID':'Item_id'})
df['Description'] = df['Description'].str.title()
df.head()

,item_id,Description,2020_units_sold,2021_start_stock
0,82486,3 Drawer Antique White Wood Cabinet,440,917
1,23435,3 Raffia Ribbons Vintage Christmas,692,1033
2,85034B,3 White Choc Morris Boxed Candles,1610,1142
3,84559A,3D Sheet Of Dog Stickers,918,620
4,23697,A Pretty Thank You Card,557,530


In [ ]:
pip install psycopg2-binary sqlalchemy

In [29]:
from sqlalchemy import create_engine, event
import urllib
server = 'DESKTOP-5KLJG2D' 
database = 'Inventory_Analytics_Engineer'
table_name = 'Stock'

# 2. Build Connection String 
# Note: We use "ODBC Driver 17" as it handles Pandas data types much better
connection_str = (
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={server};'
    f'DATABASE={database};'
    f'Trusted_Connection=yes;'
)
params = urllib.parse.quote_plus(connection_str)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
@event.listens_for(engine, "before_cursor_execute")
def receive_before_cursor_execute(conn, cursor, statement, parameters, context, executemany):
    if executemany:
        cursor.fast_executemany = True

# 5. Execute the Upload
try:
    print(f"Uploading {len(df)} rows to table [{table_name}]...")
    
    # if_exists='replace' will drop the old table and create a new one
    df.to_sql(name=table_name, con=engine, if_exists='replace', index=False)
    
    print("✅ Success! Your data is now in SQL Server.")
except Exception as e:
    print("❌ Error during upload:")
    print(e)

Uploading 104 rows to table [Stock]...
✅ Success! Your data is now in SQL Server.


##### Cost table 
###### 1- I need to rename column name to easy the integretion for me
###### 2- There is incorrect format for a column (factory_equipment_rent) 

In [38]:
df_cost = pd.read_csv(r"F:\Z_SQL&PowerBi Projects\Inventory\Costs.csv")
df_cost.head()

,SKU,raw_material,factory_labor,factory_equipment_rent,distribution,advertisement
0,10125,0.39,0.26,0..19,1.241,1.7340
1,15030,0.01,0.13,0..15,0.191,0.3596
2,16054,0.01,0.03,0..08,0.174,0.2436
3,17038,0.01,0.02,0..04,0.046,0.0868
4,20719,0.48,0.37,0..01,1.207,1.7000


In [ ]:
df_cost.info() # factory_equipment_rent should be float not object => there is incorrect data
df_cost['factory_equipment_rent'] = df_cost['factory_equipment_rent'].str.replace('..', '.')
df_cost['factory_equipment_rent'] = df_cost['factory_equipment_rent'].astype(float)



In [47]:
df_cost = df_cost.rename(columns = {'SKU' : 'Item_id'})
df_cost.head()

,Item_id,raw_material,factory_labor,factory_equipment_rent,distribution,advertisement
0,10125,0.39,0.26,0.19,1.241,1.7340
1,15030,0.01,0.13,0.15,0.191,0.3596
2,16054,0.01,0.03,0.08,0.174,0.2436
3,17038,0.01,0.02,0.04,0.046,0.0868
4,20719,0.48,0.37,0.01,1.207,1.7000


##### Orders_table
###### 1- Country code is incorrected i fixed it by spilt the column and removed the unwanted part
###### 2- there are columns i need to rename it to faclite for me integretion
###### 3- there is inconsistent data in country column in (Spain , Spain. ) i fixed it

In [56]:
df_orders = pd.read_csv(r"F:\Z_SQL&PowerBi Projects\Inventory\Orders_data.csv")
df_orders.head()

,InvoiceNo,SKU,Quantity,InvoiceDate,Country
0,539639,20932,1,1/1/2021,14083_Australia
1,539659,22677,20,1/2/2021,15034_Australia
2,540027,84849B,12,1/16/2021,14210_Australia
3,540094,21672,12,1/16/2021,14221_Australia
4,540247,22637,3,1/17/2021,15464_Australia


In [57]:
df_orders =df_orders.rename(columns = {'SKU' : 'Item_id'})

In [ ]:
df_orders.info()

In [ ]:
df_orders['Country'].value_counts(dropna = False)

In [71]:
df_orders[['NotNeed','Country']] = df_orders['Country'].str.split('_', expand = True)


In [74]:
df_orders = df_orders.drop(columns = ['NotNeed'])
df_orders.head()

,InvoiceNo,Item_id,Quantity,InvoiceDate,Country
0,539639,20932,1,1/1/2021,Australia
1,539659,22677,20,1/2/2021,Australia
2,540027,84849B,12,1/16/2021,Australia
3,540094,21672,12,1/16/2021,Australia
4,540247,22637,3,1/17/2021,Australia


In [79]:
df_orders['Country'] = df_orders['Country'].str.replace('.','')
df_orders['Country'].nunique()

35

##### Countries table

In [3]:
df_contries = pd.read_csv(r"F:\Z_SQL&PowerBi Projects\Inventory\Countries.csv")
df_contries.head()

,InvoiceNo,Country
0,536592,United Kingdom
1,536592,Netherlands
2,536544,United Kingdom
3,536544,France
4,536592,United Kingdom


In [65]:
df_contries['Country'].unique()

array(['United Kingdom', 'Netherlands', 'France', 'Israel', 'Cyprus',
       'Germany', 'Belgium', 'Australia', 'Norway', 'EIRE', 'Austria',
       'Spain', 'Switzerland', 'Sweden', 'Malta', 'Iceland',
       'Unspecified', 'Portugal', 'Finland', 'Hong Kong', 'Denmark',
       'USA', 'Singapore', 'Channel Islands', 'Italy',
       'European Community', 'RSA', 'Bahrain', 'Japan', 'Lebanon',
       'United Arab Emirates', 'Czech Republic', 'Canada', 'Greece',
       'Poland'], dtype=object)

In [ ]:
df_contries['Country'].nunique()
df_contries['Country'].value_counts(dropna = False)

##### Categories table
###### 1- split column to match with the same data in the other tables and rename it 
###### 2- there are inconsistent data in category column i fixed it 

In [92]:
df_categories = pd.read_csv(r"F:\Z_SQL&PowerBi Projects\Inventory\categories.csv")
df_categories.head()

,ID,category
0,SKU-10125,office & school
1,SKU-15030,office & school
2,SKU-16054,office & school
3,SKU-17038,office & school
4,SKU-20719,home acce


In [93]:
df_categories = df_categories.rename(columns = {'ID': 'Item_id'})
df_categories[['NotNeed','Item_id']] = df_categories['Item_id'].str.split('-',expand = True)

In [94]:
df_categories = df_categories.drop(columns = ['NotNeed'])

In [96]:

df_categories['category'] = df_categories['category'].str.replace('home accessories','home acce')
df_categories['category'] = df_categories['category'].str.replace('toys & edibles','toys')
# ---------------------
df_categories['category'] = df_categories['category'].str.replace('home acce','home accessories')
df_categories['category'] = df_categories['category'].str.replace('toys','toys & edibles')

df_categories['category'].unique()

array(['office & school', 'home accessories', 'decoration',
       'toys & edibles', 'jewelry'], dtype=object)

In [ ]:
df_categories.head()

##### Customers table
###### 1- Remove NUll values from Customers which doesn't have meaning 
###### 2- Change datatype for the column customer id to be more readable

In [2]:
df_customers = pd.read_csv(r"F:\Z_SQL&PowerBi Projects\Inventory\Customers_data.csv")
df_customers.head()

,CustomerID,InvoiceDate
0,17069.0,1/1/2021
1,NaN,1/1/2021
2,17085.0,1/1/2021
3,12721.0,1/1/2021
4,16422.0,1/1/2021


In [115]:
df_customers = df_customers[df_customers['CustomerID'].notna()]

In [117]:
df_customers['CustomerID'] = df_customers['CustomerID'].astype(int)
df_customers.head()

,CustomerID,InvoiceDate
0,17069,1/1/2021
2,17085,1/1/2021
3,12721,1/1/2021
4,16422,1/1/2021
5,15312,1/1/2021


##### Price table
###### 1- the data in column price is incorrect and i fixed it

In [118]:
df_price = pd.read_csv(r"F:\Z_SQL&PowerBi Projects\Inventory\Price.csv")
df_price.head()

,ID,Retail_Price
0,10125,1.9$$
1,15030,1.11$$
2,16054,0.87$$
3,17038,0.8$$
4,20719,1.9$$


In [119]:
df_price['Retail_Price'] = df_price['Retail_Price'].str.replace('$$','')

In [126]:
df_price = df_price.rename(columns = {'ID': 'Item_id'})

In [127]:
df_price['Retail_Price'] = df_price['Retail_Price'].astype(float)
df_price.head()

,Item_id,Retail_Price
0,10125,1.90
1,15030,1.11
2,16054,0.87
3,17038,0.80
4,20719,1.90


In [128]:
from sqlalchemy import create_engine, event
import urllib

server = 'DESKTOP-5KLJG2D' 
database = 'Inventory_Analytics_Engineer'

connection_str = (
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={server};'
    f'DATABASE={database};'
    f'Trusted_Connection=yes;'
)

params = urllib.parse.quote_plus(connection_str)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

@event.listens_for(engine, "before_cursor_execute")
def receive_before_cursor_execute(conn, cursor, statement, parameters, context, executemany):
    if executemany:
        cursor.fast_executemany = True


tables = {
    "Countries": df_contries,
    "Categories": df_categories,
    "Customers": df_customers,
    "Prices": df_price,
    "Orders":df_orders,
    "Costs":df_cost
}


for table_name, df in tables.items():
    try:
        print(f"Uploading {len(df)} rows to table [{table_name}]...")

        df.to_sql(
            name=table_name,
            con=engine,
            if_exists='replace',   # replace old table
            index=False
        )

        print(f"✅ Success! [{table_name}] uploaded.")

    except Exception as e:
        print(f"❌ Error while uploading [{table_name}]: {e}")


Uploading 13333 rows to table [Countries]...
✅ Success! [Countries] uploaded.
Uploading 104 rows to table [Categories]...
✅ Success! [Categories] uploaded.
Uploading 6313 rows to table [Customers]...
✅ Success! [Customers] uploaded.
Uploading 104 rows to table [Prices]...
✅ Success! [Prices] uploaded.
Uploading 13333 rows to table [Orders]...
✅ Success! [Orders] uploaded.
Uploading 106 rows to table [Costs]...
✅ Success! [Costs] uploaded.
